In [1]:
import sys, os

dir = os.path.abspath('..')
sys.path.append(os.path.join(dir))
while not os.path.basename(dir) == 'v2':
    parent = os.path.dirname(dir)
    if parent == dir:
        raise FileNotFoundError("No parent directory named 'v2' found.")
    dir = parent

In [2]:
DATA_NAME = 'jorge-give-and-go-feedback'

In [3]:
from unity_utils import UnityTranslator

DATA_DIR = dir + f'/data/{DATA_NAME}'
demos = UnityTranslator.get_from(DATA_DIR, sample_rate=.5)

# print(f'Number of frames: {[len(d.video.frame_dir) for d in demos]}')

Importing:   0%|          | 0/1 [00:00<?, ?it/s]

FPS: 60.0
Samplerate: 0.5
Total # of frames: 2443
Duration of Video: 40.71666666666667 seconds


Importing: 100%|██████████| 1/1 [00:05<00:00,  5.14s/it]

Total # of frames saved: 82


In [4]:
from script_utils_gemini import ScriptGenerator
from scenic_fc.api import api
generator = ScriptGenerator()
script = generator.run(demos, api = api)

Generating comprehensive scripts: 100%|██████████| 1/1 [00:31<00:00, 31.57s/it]


In [5]:
script

[[{'second': 0,
   'description': 'The video starts with a soccer field depicted as a grid. The scene is labeled "Scenic Called Pause...Everything paused except ego". The Coach (blue player) is positioned at y=13, x=0. The Opponent (red player) is at y=5, x=0. The Teammate (blue player) is at y=0, x=0, and has possession of the Ball. A text box on the left side of the screen displays "teammate receives or gets possession of the ball".'},
  {'second': 1,
   'description': 'The text box on the left side of the screen continues to display "teammate receives or gets possession of the ball". A white \'X\' cursor briefly appears at coordinates x=3, y=5 on the field grid before disappearing.'},
  {'second': 2,
   'description': 'The text box on the left side of the screen continues to display "teammate receives or gets possession of the ball". No significant visual changes occur.'},
  {'second': 3,
   'description': 'The text box on the left side of the screen continues to display "teammate r

In [3]:
DATA_NAME = 'jorge-give-and-go'

In [7]:
DATA_DIR = dir + f'/data/{DATA_NAME}'
origin_demos = UnityTranslator.get_from(DATA_DIR, sample_rate=1)

Importing:   0%|          | 0/3 [00:00<?, ?it/s]

FPS: 5.0
Samplerate: 1
Total # of frames: 327
Duration of Video: 65.4 seconds


Importing:  33%|███▎      | 1/3 [00:02<00:05,  2.94s/it]

Total # of frames saved: 66
FPS: 5.0
Samplerate: 1
Total # of frames: 243
Duration of Video: 48.6 seconds


Importing:  67%|██████▋   | 2/3 [00:05<00:02,  2.47s/it]

Total # of frames saved: 49
FPS: 5.0
Samplerate: 1
Total # of frames: 199
Duration of Video: 39.8 seconds


Importing: 100%|██████████| 3/3 [00:06<00:00,  2.26s/it]

Total # of frames saved: 40


In [2]:
SYNTHESIZED = 'jorge-give-and-go_Gemini_synth'
EXAMPLE_FSM = 'example'


In [3]:
import json
from fix_utils_gemini import Fix_Gemini
from nlp_utils_gemini import Chat

with open(f"exports/{SYNTHESIZED}.json", "r") as f:
    fsm_json = json.load(f)

with open(f"{EXAMPLE_FSM}.json", "r") as f:
    example_fsm = json.load(f)

In [4]:
feedback = "when moving you should move to the side of the field to create angle of pass to recieve ball from the teammate.",
# feedback = "Coach shouldn't move to C5, they should go to D2."

In [5]:
from scenic_fc.api import api
from fix_gemini_simple import Fix_Gemini_Simple
simple_fixer = Fix_Gemini_Simple(fsm_json, feedback = feedback, api = api)
fsm_fixed, raw_response = simple_fixer.run()

In [61]:
raw_response

'### Explanation:\n\nThe instructor\'s feedback was focused on a specific part of the FSM where the coach was inappropriately moving to the zone "C5". Instead, the coach should be moving to "D2". To correct this, I have updated the relevant constraints in the FSM for the "MoveTo" action in node with `id: "0"`, ensuring that both the `target` and `termination` constraints reflect this updated movement target.\n\n### Changes Made:\n1. **Node 0 (MoveTo Action) Target and Termination Constraints:**\n   - Updated the `zone` parameter in the `InZone` constraint from `"C5"` to `"D2"` in both the `target` and `termination` sections of the node since the coach should be moving to the new specified zone.\n\nHere is the corrected FSM:\n\n```json\n{\n  "nodes": [\n    {\n      "target": {\n        "info": "Coach moves to a position to create a favorable angle for receiving a pass from a teammate.",\n        "logic": "A1",\n        "map": [\n          "A1"\n        ],\n        "constraints": [\n   

In [62]:
fsm_fixed

{'nodes': [{'target': {'info': 'Coach moves to a position to create a favorable angle for receiving a pass from a teammate.',
    'logic': 'A1',
    'map': ['A1'],
    'constraints': [{'id': 'A1',
      'constraint': 'InZone',
      'args': {'obj': 'coach', 'zone': 'D2'}}],
    'reasoning': "The termination condition for the MoveTo action is that the Coach moves into a tactical position—a favorable zone for receiving a pass from a teammate. The demonstration images indicate that the Coach reaches a specific zone on the field where the angle for receiving a pass is optimal. We model this future condition using the InZone API by checking whether the Coach is in the 'FavorableZone', a zone value to be learned from demonstration data and corrected to 'D2' as per feedback."},
   'id': '0',
   'action': 'MoveTo',
   'info': 'Coach moves to a position to create a favorable angle for receiving a pass from a teammate.',
   't': {'demo_0': 10, 'demo_1': 5},
   'synthesized': True,
   'terminatio

In [10]:
from scenic_fc.api import api
from fix_utils_gemini import Fix_Gemini
fixer = Fix_Gemini(fsm_json, script, origin_demos, api = api, example_fsm = example_fsm)
fsm_fixed, raw_response = fixer.run()

In [11]:
fsm_fixed

{'nodes': [{'target': {'info': 'Move to an open space to be available for a pass from the teammate.',
    'logic': 'A1',
    'map': ['A1'],
    'constraints': [{'id': 'A1',
      'constraint': 'DistanceTo',
      'args': {'from': 'opponent',
       'to': 'Coach',
       'min': {'avg': 5.921706745077121, 'std': 0.18424753231768948},
       'max': None,
       'operator': 'greater_than'}}]},
   'id': '0',
   'action': 'MoveTo',
   'info': 'Move to an open space to be available for a pass from the teammate.',
   't': {'demo_0': 20, 'demo_1': 4, 'demo_2': 11},
   'synthesized': True,
   'termination': {'info': 'Coach is in a suitable position to receive a pass from the teammate.',
    'logic': 'A1',
    'map': ['A1'],
    'constraints': [{'id': 'A1',
      'constraint': 'DistanceTo',
      'args': {'from': 'opponent',
       'to': 'Coach',
       'min': {'avg': 5.921706745077121, 'std': 0.18424753231768948},
       'max': None,
       'operator': 'greater_than'}}]}},
  {'id': '1',
   'acti

In [33]:
fsm_json

{'nodes': [{'target': {'info': 'Move to an open space to be available for a pass from the teammate.',
    'logic': 'A1',
    'map': ['A1'],
    'constraints': [{'id': 'A1',
      'constraint': 'DistanceTo',
      'args': {'from': 'opponent',
       'to': 'Coach',
       'min': {'avg': 5.921706745077121, 'std': 0.18424753231768948},
       'max': None,
       'operator': 'greater_than'}}],
    'reasoning': "To be in an open space for receiving a pass, the coach must avoid the opponent’s defensive pressure. In soccer, an 'open space' typically means that the coach is positioned far enough away from the opponent so that they are not immediately marked. Thus, we select the DistanceTo API with the 'greater_than' operator to ensure that the distance between the Coach and the Opponent is larger than a learned threshold, providing a safe zone for receiving a pass from the teammate."},
   'id': '0',
   'action': 'MoveTo',
   'info': 'Move to an open space to be available for a pass from the tea

In [13]:
raw_response

'To fix the FSM, we need to address two main issues:\n\n1. The sequence where the opponent pressures the teammate should be explicitly represented before the pass to the coach.\n2. After the coach receives the ball, ensure the FSM indicates the option to approach the goal instead of passing back, reflecting the scenarios where the opponent doesn\'t pressure the coach.\n\n### Changes Needed:\n\n1. Update the transition conditions between nodes to represent the scenario where the opponent pressures the teammate.\n2. Modify the conditions and actions post-receiving for the coach, indicating when to approach the goal directly.\n\nHere\'s the corrected FSM:\n\n```json\n{\n  "nodes": [\n    {\n      "target": {\n        "info": "Move to an open space to be available for a pass from the teammate.",\n        "logic": "A1",\n        "map": [\n          "A1"\n        ],\n        "constraints": [\n          {\n            "id": "A1",\n            "constraint": "DistanceTo",\n            "args": {

In [ ]:
with open(f'exports/{SYNTHESIZED}_fixed_fsm.json', 'w') as f:
        json.dump(fsm_fixed, f, indent=4)